# Data Validation implemented

### Imports and Configurations

In [0]:
from pyspark.sql import functions as F

In [0]:
# this is comment to test branch feature this is updated on feature branch
catalog = "ecommerce"
bronze_schema = "bronze"

datasets = [
    "customers",
    "products",
    "orders",
    "payments",
    "returns"
]

### SECTION 1 — Check Bronze Tables

In [0]:
spark.sql(f"SHOW TABLES IN {catalog}.{bronze_schema}").show(truncate=False)

### SECTION 2 — Record Counts

In [0]:
print("=" * 50)
print("BRONZE RECORD COUNTS")
print("=" * 50)

for dataset in datasets:

    table = f"{catalog}.{bronze_schema}.{dataset}"

    count = spark.table(table).count()

    print(f"{dataset:<12} : {count}")

### SECTION 3 — Schema Validation

In [0]:
for dataset in datasets:

    print("\n")
    print("=" * 70)
    print(dataset.upper())
    print("=" * 70)

    spark.table(
        f"{catalog}.{bronze_schema}.{dataset}"
    ).printSchema()

### SECTION 4 — Preview Data

In [0]:
for dataset in datasets:

    print("\n")
    print("=" * 60)
    print(dataset.upper())
    print("=" * 60)

    display(
        spark.table(f"{catalog}.{bronze_schema}.{dataset}")
        .limit(10)
    )

### SECTION 5 — Null Checks

In [0]:
# customers
customers = spark.table("ecommerce.bronze.customers")

customers.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in customers.columns
    ]
).show()

In [0]:
# products
products = spark.table("ecommerce.bronze.products")

products.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in products.columns
    ]
).show()

In [0]:
# orders
orders = spark.table("ecommerce.bronze.orders")

orders.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in orders.columns
    ]
).show()

In [0]:
# payments
payments = spark.table("ecommerce.bronze.payments")

payments.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in payments.columns
    ]
).show()

In [0]:
# returns
returns = spark.table("ecommerce.bronze.returns")

returns.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in returns.columns
    ]
).show()

### SECTION 6 — Duplicate Checks

In [0]:
duplicate_columns = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "payments": "payment_id",
    "returns": "return_id"
}

for dataset, pk in duplicate_columns.items():

    print("\n")
    print("=" * 70)
    print(dataset.upper())
    print("=" * 70)

    duplicates = (
        spark.table(f"ecommerce.bronze.{dataset}")
             .groupBy(pk)
             .count()
             .filter("count > 1")
    )

    print("Duplicate Records :", duplicates.count())

    duplicates.show()

### SECTION 7 — Descriptive Statistics

In [0]:
numeric_tables = [
    "products",
    "orders",
    "payments",
    "returns"
]

for table in numeric_tables:

    print("\n")
    print("=" * 70)
    print(table.upper())
    print("=" * 70)

    # 1. Load the DataFrame
    df = spark.table(f"ecommerce.bronze.{table}")
    
    # 2. Get the descriptive statistics
    df_summary = df.describe()
    
    # 3. Use display() to render a clean, interactive table
    display(df_summary)

### SECTION 8 — SQL Validation

In [0]:
%sql
SELECT COUNT(*) AS total_customers
FROM ecommerce.bronze.customers;

In [0]:
%sql
SELECT country,
       COUNT(*) AS customers
FROM ecommerce.bronze.customers
GROUP BY country
ORDER BY customers DESC;

In [0]:
%sql
SELECT category,
       COUNT(*) AS products
FROM ecommerce.bronze.products
GROUP BY category
ORDER BY products DESC;

In [0]:
%sql
SELECT payment_method,
       COUNT(*) AS total
FROM ecommerce.bronze.payments
GROUP BY payment_method;

In [0]:
%sql
SELECT return_reason,
       COUNT(*) AS total_returns
FROM ecommerce.bronze.returns
GROUP BY return_reason;

### SECTION 9 — Relationship Validation

Every order should belong to a customer. Expected:

0

In [0]:
%sql
SELECT COUNT(*) AS invalid_orders
FROM ecommerce.bronze.orders o
LEFT JOIN ecommerce.bronze.customers c
ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

Every payment should belong to an order.

In [0]:
%sql
SELECT COUNT(*) AS invalid_payments
FROM ecommerce.bronze.payments p
LEFT JOIN ecommerce.bronze.orders o
ON p.order_id = o.order_id
WHERE o.order_id IS NULL;

Every return should belong to an order.

In [0]:
%sql
SELECT COUNT(*) AS invalid_returns
FROM ecommerce.bronze.returns r
LEFT JOIN ecommerce.bronze.orders o
ON r.order_id = o.order_id
WHERE o.order_id IS NULL;

### SECTION 10 — Bronze Summary Report

In [0]:
summary = []

for dataset, pk in duplicate_columns.items():

    df = spark.table(f"ecommerce.bronze.{dataset}")

    total = df.count()

    duplicate = (
        df.groupBy(pk)
          .count()
          .filter("count > 1")
          .count()
    )

    summary.append(
        (
            dataset,
            total,
            duplicate
        )
    )

summary_df = spark.createDataFrame(
    summary,
    [
        "Table",
        "Total Records",
        "Duplicate Keys"
    ]
)

display(summary_df)